# 01 EDA

Exploratory analysis notebook for raw transaction and merchant data.

In [ ]:
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR

In [ ]:
import polars as pl

# 1. Список топ-20 MCC кодов, которые нужно проанализировать
top20_mcc = [
    "8651", "5942", "4011", "3958", "3700", "3724", "3245", "3828", "3612", "5733",
    "3904", "8699", "3924", "3182", "3774", "3578", "3928", "6011", "3503", "3235"
]

biz_absolute_path = r"...\business_cards_MDQ.parquet"
cons_absolute_path = r"...\consumer_cards_MDQ.parquet"

df_biz = pl.read_parquet(biz_absolute_path).with_columns(pl.lit(1).alias("target"))
df_cons = pl.read_parquet(cons_absolute_path).with_columns(pl.lit(0).alias("target"))


df_trans = pl.concat([df_biz, df_cons], how="vertical")

mcc_col = "mcc" if "mcc" in df_trans.columns else "mcc_code"
df_trans = df_trans.with_columns(pl.col(mcc_col).cast(pl.Utf8))

mcc_analysis = (
    df_trans.filter(pl.col(mcc_col).is_in(top20_mcc))
    .group_by(mcc_col)
    .agg([
        pl.len().alias("total_count"),
        pl.col("target").sum().alias("business_count"),
        (pl.col("target") == 0).sum().alias("consumer_count"),
    ])
    .with_columns(
        (pl.col("business_count") / pl.col("total_count")).alias("business_share")
    )
    .sort("business_share", descending=True)
)

mcc_analysis

In [ ]:
import polars as pl
import numpy as np


biz_absolute_path = r"...\business_cards_MDQ.parquet"
cons_absolute_path = r"...\consumer_cards_MDQ.parquet"

# 1. Читаем данные и маркируем таргет
df_biz = pl.read_parquet(biz_absolute_path).with_columns(pl.lit(1).alias("target"))
df_cons = pl.read_parquet(cons_absolute_path).with_columns(pl.lit(0).alias("target"))
df_all_trans = pl.concat([df_biz, df_cons], how="vertical")

# Приводим MCC к строке
df_all_trans = df_all_trans.with_columns(pl.col("mcc").cast(pl.Utf8))

# 2. ВАРИАНТ 2: Считаем B2B-индекс для каждого MCC (доля бизнеса в этом коде)
mcc_weights = (
    df_all_trans.group_by("mcc")
    .agg([
        pl.col("target").mean().alias("mcc_b2b_weight")
    ])
)

# Подклеиваем веса обратно к транзакциям
df_all_trans = df_all_trans.join(mcc_weights, on="mcc", how="left")

# 3. ВАРИАНТ 1: Агрегаты сумм трат для бытовых MCC (супермаркеты, рестораны, такси)
# Считаем для каждой карты её общие показатели и специфичные чеки
cards_features = (
    df_all_trans.group_by("card_number")
    .agg([
        # Общий таргет карты (1 или 0)
        pl.col("target").first().alias("target"),
        
        # Общие фичи по суммам
        pl.col("transaction_amount_kzt").max().alias("global_max_amount"),
        pl.col("transaction_amount_kzt").mean().alias("global_mean_amount"),
        pl.col("transaction_amount_kzt").quantile(0.95).alias("global_quantile_95"),
        
        # Средний B2B-индекс по всем транзакциям карты (Вариант 2)
        pl.col("mcc_b2b_weight").mean().alias("card_b2b_index"),
        
        # Фичи по конкретным бытовым MCC (Вариант 1)
        # 5411 - Супермаркеты
        pl.col("transaction_amount_kzt").filter(pl.col("mcc") == "5411").max().alias("max_amount_mcc_5411"),
        pl.col("transaction_amount_kzt").filter(pl.col("mcc") == "5411").sum().alias("sum_amount_mcc_5411"),
        
        # 5812 - Рестораны и кафе
        pl.col("transaction_amount_kzt").filter(pl.col("mcc") == "5812").max().alias("max_amount_mcc_5812"),
        
        # 4121 - Такси
        pl.col("transaction_amount_kzt").filter(pl.col("mcc") == "4121").sum().alias("sum_amount_mcc_4121")
    ])
    # Заполняем нулями те МСС, по которым у карты не было трат
    .fill_null(0)
)

# 4. Проверяем, как фичи разделяют бизнес и физиков
print("Сравнение средних значений признаков:")
print(
    cards_features.group_by("target")
    .agg([
        pl.col("global_max_amount").mean().alias("ср_макс_чек"),
        pl.col("global_quantile_95").mean().alias("ср_95_процентиль"),
        pl.col("card_b2b_index").mean().alias("ср_b2b_индекс"),
        pl.col("max_amount_mcc_5411").mean().alias("ср_макс_в_супермаркете")
    ])
)

In [ ]:
import polars as pl

biz_absolute_path = r"...\business_cards_MDQ.parquet"
cons_absolute_path = r"...\consumer_cards_MDQ.parquet"

# 1. Загрузка
df_biz = pl.read_parquet(biz_absolute_path).with_columns(pl.lit(1).alias("target"))
df_cons = pl.read_parquet(cons_absolute_path).with_columns(pl.lit(0).alias("target"))
df_all_trans = pl.concat([df_biz, df_cons], how="vertical")

df_all_trans = df_all_trans.with_columns([
    pl.col("mcc").cast(pl.Utf8),
    # Достаем час транзакции из таймстемпа для временных фичей
    pl.col("transaction_timestamp").dt.hour().alias("trans_hour")
])

# 2. Пересчитываем базовый b2b-весовый индекс для MCC
mcc_weights = (
    df_all_trans.group_by("mcc")
    .agg([pl.col("target").mean().alias("mcc_b2b_weight")])
)
df_all_trans = df_all_trans.join(mcc_weights, on="mcc", how="left")

# 3. Собираем фичи по степам (Деньги + Время + Каналы)
extended_features = (
    df_all_trans.group_by("card_number")
    .agg([
        pl.col("target").first().alias("target"),
        
        # Блок 1: Деньги (уже проверенные)
        pl.col("transaction_amount_kzt").max().alias("global_max_amount"),
        pl.col("transaction_amount_kzt").quantile(0.95).alias("global_quantile_95"),
        pl.col("mcc_b2b_weight").mean().alias("card_b2b_index"),
        
        # Блок 2: Время (Новые) 
        # Доля транзакций, совершенных глубокой ночью (с 00:00 до 05:59)
        (pl.col("trans_hour").is_between(0, 5)).mean().alias("night_trans_share"),
        # Доля транзакций в стандартное рабочее время (с 09:00 до 18:00)
        (pl.col("trans_hour").is_between(9, 18)).mean().alias("working_hours_share"),
        
        # Блок 3: Каналы и Типы (Новые фичи)
        # Доля онлайн операций
        (pl.col("channel") == "online").mean().alias("online_share"),
        # Доля регулярных платежей/подписок
        (pl.col("is_recurring") == True).mean().alias("recurring_share")
    ])
    .fill_null(0)
)

# 4. Проверяем новые признаки в разрезе бизнес / физик
print("Проверка расширенного пакета фичей:")
print(
    extended_features.group_by("target")
    .agg([
        pl.col("night_trans_share").mean().alias("доля_ночных_трат"),
        pl.col("working_hours_share").mean().alias("доля_в_рабочее_время"),
        pl.col("online_share").mean().alias("доля_онлайн"),
        pl.col("recurring_share").mean().alias("доля_подписок")
    ])
)

In [ ]:
import polars as pl

# 1. Функция для сборки безопасных фичей по транзакциям
def build_safe_features(df_train_trans: pl.DataFrame, df_test_trans: pl.DataFrame):
    mcc_weights = (
        df_train_trans.with_columns(pl.col("mcc").cast(pl.Utf8))
        .group_by("mcc")
        .agg([pl.col("target").mean().alias("mcc_b2b_weight")])
    )
    
    def _aggregate_features(df_trans, weights):
        df_proc = df_trans.with_columns([
            pl.col("mcc").cast(pl.Utf8),
            pl.col("transaction_timestamp").dt.hour().alias("trans_hour")
        ]).join(weights, on="mcc", how="left").fill_null(0.0)
        
        return (
            df_proc.group_by("card_number")
            .agg([
                pl.col("target").first().alias("target") if "target" in df_proc.columns else pl.lit(None).alias("target"),
                pl.col("transaction_amount_kzt").max().alias("global_max_amount"),
                pl.col("transaction_amount_kzt").quantile(0.95).alias("global_quantile_95"),
                pl.col("mcc_b2b_weight").mean().alias("card_b2b_index"),
                (pl.col("trans_hour").is_between(0, 5)).mean().alias("night_trans_share"),
                (pl.col("trans_hour").is_between(9, 18)).mean().alias("working_hours_share"),
                (pl.col("channel") == "online").mean().alias("online_share"),
                (pl.col("is_recurring") == True).mean().alias("recurring_share")
            ])
            .fill_null(0)
        )
    
    train_features = _aggregate_features(df_train_trans, mcc_weights)
    test_features = _aggregate_features(df_test_trans, mcc_weights)
    return train_features, test_features


# 2. Тест
biz_absolute_path = r"...\business_cards_MDQ.parquet"
cons_absolute_path = r"...\consumer_cards_MDQ.parquet"

df_biz = pl.read_parquet(biz_absolute_path).with_columns(pl.lit(1).alias("target"))
df_cons = pl.read_parquet(cons_absolute_path).with_columns(pl.lit(0).alias("target"))

train_feats, test_feats = build_safe_features(df_biz, df_cons)

print("Трейн фичи собрались, размер:", train_feats.shape)
print("Тест фичи собрались, размер:", test_feats.shape)

In [ ]:
print(df_all_trans.select("transaction_timestamp").head(5))

In [ ]:
# Смотрим на разброс сумм и уникальность MCC для нескольких карт для проверки адекватности агрегаций
check_df = (
    df_all_trans.group_by("card_number")
    .agg([
        pl.col("mcc").n_unique().alias("unique_mcc_count"),
        pl.col("transaction_amount_kzt").count().alias("total_trans"),
        pl.col("transaction_amount_kzt").std().alias("amount_std"),
        pl.col("transaction_amount_kzt").mean().alias("amount_mean"),
    ])
    .head(10)
)

print(check_df)